# QuickSort

**Sortare in-place**: un algoritm care nu folosește memorie auxiliară de ordinul O(n). Este permisă memorie o(n), de exemplu O(log n) sau O(√n).
- MergeSort **nu** este in-place: interclasarea folosește un vector auxiliar de lungime n.
- HeapSort și QuickSort sunt in-place.

QuickSort folosește tot **Divide et Impera**, dar invers față de MergeSort:

| | MergeSort | QuickSort |
|---|---|---|
| Împarte | trivial: la mijloc | **partiționare** – O(n) |
| Combină | **interclasare** – O(n) | trivial: nu e nimic de făcut |

Pași:
1. Alegem un **pivot**.
2. **Partiționăm**: elementele mai mici decât pivotul ajung în stânga, cele mai mari în dreapta.
3. Sortăm recursiv cele două părți.

Proprietăți:
- **in-place** (doar stiva recursivității: O(log n) în medie)
- **nu este stabil**
- **O(n log n)** în medie și în cazul favorabil, **O(n²)** în cazul cel mai nefavorabil

## 1. Partiționarea Lomuto

Pivotul este **ultimul** element, `p = a[hi]`. Parcurgem cu `j` de la `lo` la `hi - 1` și menținem invariantul:
- `a[lo .. i-1]` < p
- `a[i .. j-1]` ≥ p
- `a[j .. hi-1]` – încă neprocesate

Când `a[j] < p`, îl aducem în zona din stânga (swap cu `a[i]`) și creștem `i`. La final punem pivotul pe poziția `i`, care este **locul lui final** în vectorul sortat.

In [ ]:
# pivot = a[hi], returneaza pozitia finala a pivotului
def partition_lomuto(a, lo, hi):
    p = a[hi]
    i = lo
    for j in range(lo, hi):
        if a[j] < p:
            a[i], a[j] = a[j], a[i]
            i += 1
    a[i], a[hi] = a[hi], a[i]       # pivotul ajunge pe locul lui final
    return i


a = [5, 3, 8, 4, 2, 7, 1, 6]
k = partition_lomuto(a, 0, len(a) - 1)
print(a, "-> pivotul", a[k], "e pe poziția", k)

Pas cu pas:

In [ ]:
def partition_lomuto_verbose(a, lo, hi):
    p = a[hi]
    i = lo
    print(f"pivot = {p}, a = {a}")
    for j in range(lo, hi):
        x = a[j]
        if x < p:
            a[i], a[j] = a[j], a[i]
            print(f"j = {j}: {x} <  {p} -> swap(a[{i}], a[{j}]), a = {a}, i = {i + 1}")
            i += 1
        else:
            print(f"j = {j}: {x} >= {p} -> nimic")
    a[i], a[hi] = a[hi], a[i]
    print(f"pivotul pe poziția {i}:  a = {a}")
    return i


partition_lomuto_verbose([5, 3, 8, 4, 2, 7, 1, 6], 0, 7)

## 2. Partiționarea Hoare

Pivotul este elementul din **mijloc**, ca în curs (`(st + dr) / 2`). Doi indici pornesc din capete și se apropie:
- `i` avansează cât timp `a[i] < p`
- `j` coboară cât timp `a[j] > p`
- dacă `i < j`, interschimbăm `a[i]` cu `a[j]` și continuăm; altfel ne oprim și întoarcem `j`.

Rezultat: `a[lo..j]` ≤ p ≤ `a[j+1..hi]`.

⚠️ Diferențe față de Lomuto:
- pivotul **nu** ajunge neapărat pe locul lui final, deci recursivitatea se face pe `[lo, j]` și `[j+1, hi]` (nu pe `[lo, k-1]` și `[k+1, hi]`)
- face în medie de aproximativ 3 ori mai puține interschimbări
- se descurcă bine cu elemente egale (vezi secțiunea 5)

In [ ]:
# pivot = elementul din mijloc, returneaza j cu a[lo..j] <= p <= a[j+1..hi]
def partition_hoare(a, lo, hi):
    p = a[(lo + hi) // 2]
    i, j = lo - 1, hi + 1
    while True:
        i += 1
        while a[i] < p:
            i += 1
        j -= 1
        while a[j] > p:
            j -= 1
        if i >= j:
            return j
        a[i], a[j] = a[j], a[i]


def partition_hoare_verbose(a, lo, hi):
    p = a[(lo + hi) // 2]
    print(f"pivot = {p}, a = {a}")
    i, j = lo - 1, hi + 1
    while True:
        i += 1
        while a[i] < p:
            i += 1
        j -= 1
        while a[j] > p:
            j -= 1
        if i >= j:
            print(f"i = {i} >= j = {j} -> stop:  {a[lo:j + 1]} | {a[j + 1:hi + 1]}")
            return j
        a[i], a[j] = a[j], a[i]
        print(f"i = {i}, j = {j} -> swap, a = {a}")


partition_hoare_verbose([5, 3, 8, 4, 2, 7, 1, 6], 0, 7)

## 3. QuickSort

In [ ]:
def quicksort_lomuto(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo < hi:
        k = partition_lomuto(a, lo, hi)
        quicksort_lomuto(a, lo, k - 1)     # a[k] e deja la locul lui
        quicksort_lomuto(a, k + 1, hi)
    return a


def quicksort_hoare(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo < hi:
        j = partition_hoare(a, lo, hi)
        quicksort_hoare(a, lo, j)          # atentie: j inclus!
        quicksort_hoare(a, j + 1, hi)
    return a


print(quicksort_lomuto([38, 27, 43, 3, 9, 82, 10]))
print(quicksort_hoare([38, 27, 43, 3, 9, 82, 10]))

Urmărirea recursivității (Lomuto). Între paranteze drepte este pivotul, deja pe locul lui final:

In [ ]:
def quicksort_verbose(a, lo=0, hi=None, depth=0):
    if hi is None:
        hi = len(a) - 1
    if lo < hi:
        indent = "    " * depth
        before = a[lo:hi + 1]
        k = partition_lomuto(a, lo, hi)
        print(f"{indent}{before} -> {a[lo:k]} [{a[k]}] {a[k + 1:hi + 1]}")
        quicksort_verbose(a, lo, k - 1, depth + 1)
        quicksort_verbose(a, k + 1, hi, depth + 1)
    return a


quicksort_verbose([38, 27, 43, 3, 9, 82, 10])

## 4. Alegerea pivotului și complexitatea

Totul depinde de **unde ajunge pivotul** după partiționare.

**Cazul favorabil**: pivotul ajunge mereu la mijloc:
$$T(n) = 2\,T\!\left(\frac{n}{2}\right) + O(n) \;\Rightarrow\; O(n \log n) \quad \text{(Teorema Master, cazul 2)}$$

**Cazul nefavorabil**: pivotul este mereu minimul sau maximul (o parte are n − 1 elemente, cealaltă 0):
$$T(n) = T(n-1) + O(n) = O(n) + O(n-1) + \dots + O(1) \;\Rightarrow\; O(n^2)$$
Exemplu: Lomuto cu pivotul ultimul element, pe un vector **deja sortat**.

**Cazul general**: pivotul împarte în proporția α : (1 − α):
$$T(n) = T(\alpha n) + T\big((1-\alpha)\,n\big) + O(n)$$
Nu se rezolvă cu Teorema Master, dar se rezolvă cu arborele de recursivitate: fiecare nivel costă cel mult $cn$, iar adâncimea este $\log_{1/\max(\alpha,\,1-\alpha)} n = O(\log n)$ pentru orice α constant ⇒ **O(n log n)**. Chiar și o împărțire 1/10 – 9/10 dă O(n log n)!

**Cazul mediu** (sau cu pivot **aleator**, pentru orice intrare): O(n log n), cu aproximativ $2n \ln n \approx 1.39\, n \log_2 n$ comparații.

Variante pentru pivot:
- primul / ultimul element – simplu, dar O(n²) pe vectori deja sortați
- elementul din mijloc – bun pe vectori sortați, dar există și pentru el intrări nefavorabile
- **aleator** – nicio intrare nu mai e mereu nefavorabilă: O(n log n) în medie pentru *orice* intrare
- mediana din 3 (primul, mijlocul, ultimul) – folosită des în practică

Să numărăm comparațiile pe un vector **deja sortat** și pe unul aleator, de lungime 500:

In [ ]:
import math
import random


# quicksort cu lomuto, returneaza cate comparatii a facut
def quicksort_count(a, pivot="last"):
    comparisons = 0

    def sort(lo, hi):
        nonlocal comparisons
        if lo >= hi:
            return
        if pivot == "random":
            r = random.randint(lo, hi)
            a[r], a[hi] = a[hi], a[r]   # mutam pivotul ales la final
        elif pivot == "middle":
            m = (lo + hi) // 2
            a[m], a[hi] = a[hi], a[m]
        comparisons += hi - lo          # partitionarea face hi - lo comparatii
        k = partition_lomuto(a, lo, hi)
        sort(lo, k - 1)
        sort(k + 1, hi)

    sort(0, len(a) - 1)
    return comparisons


n = 500
print(f"n = {n}:  n²/2 = {n * n // 2},  n·log₂n = {round(n * math.log2(n))}\n")
for pivot in ["last", "middle", "random"]:
    on_sorted = quicksort_count(list(range(n)), pivot)
    on_random = quicksort_count(random.sample(range(n), n), pivot)
    print(f"pivot {pivot:6}:  vector sortat -> {on_sorted:6} comparații,  vector aleator -> {on_random:6}")

⚠️ Pe vectori sortați mari, varianta cu ultimul element ca pivot are recursivitate de adâncime n. În Python dă `RecursionError` (limita implicită e 1000), iar în C++ poate da stack overflow.

Truc: apelăm recursiv doar pe partea **mai mică**, iar pe cea mare continuăm cu un `while`. Așa adâncimea stivei e O(log n) garantat (vezi `quicksort_small_stack` în `quicksort.cpp`).

## 5. Elemente egale

Dacă toate elementele sunt egale:
- **Lomuto**: `a[j] < p` nu e niciodată adevărat ⇒ pivotul rămâne la capăt ⇒ împărțire n − 1 / 0 ⇒ **O(n²)**
- **Hoare**: `i` și `j` se opresc la fiecare pas și se întâlnesc la mijloc ⇒ împărțire echilibrată ⇒ O(n log n)

Soluția clasică este **partiționarea în 3** (problema steagului olandez, Dijkstra): `< p | == p | > p`. Recursivitatea se face doar pe zonele `< p` și `> p`.

In [ ]:
# pivot aleator, returneaza (lt, gt): a[lo..lt-1] < p, a[lt..gt] == p, a[gt+1..hi] > p
def partition_3way(a, lo, hi):
    p = a[random.randint(lo, hi)]
    lt, i, gt = lo, lo, hi
    while i <= gt:
        if a[i] < p:
            a[lt], a[i] = a[i], a[lt]
            lt += 1
            i += 1
        elif a[i] > p:
            a[i], a[gt] = a[gt], a[i]
            gt -= 1                     # nu crestem i, a[i] e nou si inca nu l-am verificat
        else:
            i += 1
    return lt, gt


def quicksort_3way(a, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo < hi:
        lt, gt = partition_3way(a, lo, hi)
        quicksort_3way(a, lo, lt - 1)
        quicksort_3way(a, gt + 1, hi)
    return a


print(quicksort_3way([3, 1, 3, 2, 3, 1, 2, 3, 3]))

Numărăm comparațiile cu un „număr” care ține minte de câte ori a fost comparat:

In [ ]:
# un numar care tine minte de cate ori a fost comparat
class Num:
    comparisons = 0

    def __init__(self, v):
        self.v = v

    def __lt__(self, other):
        Num.comparisons += 1
        return self.v < other.v

    def __gt__(self, other):
        Num.comparisons += 1
        return self.v > other.v


n = 600
for name, sort in [("Lomuto", quicksort_lomuto), ("Hoare", quicksort_hoare), ("3-way", quicksort_3way)]:
    Num.comparisons = 0
    sort([Num(7) for _ in range(n)])
    print(f"{name:7}: {Num.comparisons:7} comparații pentru {n} elemente egale")

## 6. Limita inferioară: orice sortare prin comparații face Ω(n log n) comparații

Un algoritm care sortează doar prin comparații poate fi văzut ca un **arbore de decizie**:
- fiecare nod intern este o comparație `a[i] < a[j]?`
- fiecare frunză este o permutare (ordinea finală)
- o execuție este un drum de la rădăcină la o frunză, deci numărul de comparații din cazul cel mai nefavorabil este **înălțimea h**

Algoritmul trebuie să poată produce oricare dintre cele **n!** permutări, deci arborele are cel puțin n! frunze. Un arbore binar de înălțime h are cel mult $2^h$ frunze:
$$n! \le 2^h \;\Rightarrow\; h \ge \log_2 n!$$

Cu aproximarea lui Stirling, $\log_2 n! = n \log_2 n - n \log_2 e + O(\log n) = \Theta(n \log n)$. Sau mai simplu: jumătate din factorii lui n! sunt ≥ n/2, deci $n! \ge (n/2)^{n/2}$ și $\log_2 n! \ge \frac{n}{2} \log_2 \frac{n}{2}$. În ambele cazuri:
$$h = \Omega(n \log n)$$

Exemplu: pentru n = 3 avem 3! = 6 frunze ⇒ h ≥ ⌈log₂ 6⌉ = 3 comparații.

Consecință: MergeSort și HeapSort sunt **optime asimptotic**. Ca să coborâm sub n log n trebuie să **nu** comparăm elemente (CountSort, RadixSort din C7).

In [ ]:
for n in [3, 5, 10, 100, 1000, 10**6]:
    lower_bound = math.ceil(math.lgamma(n + 1) / math.log(2))    # log2(n!)
    print(f"n = {n:>7}:  ⌈log₂ n!⌉ = {lower_bound:>9},   n·log₂n = {round(n * math.log2(n)):>9}")

## Testare

In [ ]:
for _ in range(500):
    arr = [random.randint(-20, 20) for _ in range(random.randint(0, 40))]
    assert quicksort_lomuto(list(arr)) == sorted(arr)
    assert quicksort_hoare(list(arr)) == sorted(arr)
    assert quicksort_3way(list(arr)) == sorted(arr)
    for pivot in ["last", "middle", "random"]:
        b = list(arr)
        quicksort_count(b, pivot)
        assert b == sorted(arr)
print("toate testele au trecut")